# Five Archetypes at the Top of YouTube: KMeans on the Global Top-995

The Global YouTube Statistics 2023 table holds 995 of the largest channels on the platform with subscriber counts, lifetime views, upload counts, category, country, and a pair of earnings bands. This notebook runs KMeans at k=5 on log-standardised metrics and interprets the resulting segmentation as five recognisable creator archetypes: mega-scale, mainstream, low engagement, music-video, and upload machines. Along the way it teaches why log-transform matters for skewed count data, how to read the elbow plot against silhouette, and how to go from raw cluster labels to a usable profile.


## Load the data

Kaggle publishes the dataset at `/kaggle/input/global-youtube-statistics-2023/Global YouTube Statistics.csv`. The code below falls back to a local path when the notebook is executed outside a Kaggle kernel.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

KAGGLE_PATH = Path("/kaggle/input/global-youtube-statistics-2023/Global YouTube Statistics.csv")
LOCAL_PATH = Path("data/Global YouTube Statistics.csv")
path = KAGGLE_PATH if KAGGLE_PATH.exists() else LOCAL_PATH

df = pd.read_csv(path, encoding="latin-1")
print(f"rows: {len(df)}  cols: {len(df.columns)}")
df.head(3)


### Cleaning

Three core count columns need to be numeric. Rows with zero or missing subscribers, views, or uploads cannot participate in the segmentation, so they drop here. The earnings midpoint is a derived statistic used only for the category ranking much later.


In [ ]:
for c in ["subscribers", "video views", "uploads"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in ["lowest_yearly_earnings", "highest_yearly_earnings"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
df["mid_yearly_earnings"] = (df["lowest_yearly_earnings"] + df["highest_yearly_earnings"]) / 2

df_ = df.loc[(df["subscribers"] > 0) & (df["video views"] > 0) & (df["uploads"] > 0)].copy()
print(f"rows after positive-count filter: {len(df_)}")
print(f"countries represented: {df_['Country'].nunique()}")


## Why log-transform the features

Subscribers, views, and uploads all span five or six orders of magnitude across the top-995. Without transformation, the one channel at 245M subscribers (T-Series) pulls the mean so far that distances in Euclidean space are dominated by it. KMeans minimises squared Euclidean distance to centroids, so feeding it unscaled skewed counts produces one cluster containing T-Series and one containing everyone else.

`np.log1p` compresses the scale: a channel with 245M subscribers lands at log(246M) ≈ 19.3, while one at 10M lands at 16.1. The ratio that mattered (24x apart) is now a small additive gap. Standardising afterwards centres each feature at zero with unit variance so that views-per-subscriber and raw subscribers contribute comparably to distance calculations.


In [ ]:
df_["views_per_sub"] = df_["video views"] / df_["subscribers"]
df_["views_per_upload"] = df_["video views"] / df_["uploads"]
df_["uploads_per_sub"] = df_["uploads"] / (df_["subscribers"] / 1e6)

feats = ["subscribers", "video views", "uploads", "views_per_sub", "views_per_upload", "uploads_per_sub"]

print("raw feature skew (Fisher-Pearson):")
print(df_[feats].skew().round(2))
print("\nlog1p feature skew:")
print(np.log1p(df_[feats]).skew().round(2))


The raw features are heavily right-skewed — subscribers skew at roughly 6, views per upload near 18. After `log1p`, every feature sits near symmetric. That is the distributional shape KMeans is happy working in.


## Standardise and pick k with the elbow method

The elbow method plots within-cluster SSE (inertia) as k grows. SSE falls monotonically because adding clusters always reduces the sum of squared distances to the nearest centroid. The useful signal is where the rate of decrease flattens — the "elbow". Silhouette score plotted alongside provides a second opinion: it measures how tight each cluster is relative to its neighbours, and peaks at well-separated k.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

Xc = df_[feats].copy().dropna()
X_log = np.log1p(Xc)
X_std = StandardScaler().fit_transform(X_log)

ks = list(range(2, 11))
inertias, sils = [], []
for k in ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_std)
    inertias.append(km.inertia_)
    sils.append(silhouette_score(X_std, km.labels_))

import matplotlib.pyplot as plt
fig, ax1 = plt.subplots(figsize=(9, 4.8))
ax1.plot(ks, inertias, "o-", color="#ef4444", lw=2.5, label="SSE")
ax1.set_xlabel("k")
ax1.set_ylabel("Within-cluster SSE", color="#ef4444")
ax1.tick_params(axis="y", labelcolor="#ef4444")
ax2 = ax1.twinx()
ax2.plot(ks, sils, "s--", color="#0a0a0a", lw=2, label="silhouette")
ax2.set_ylabel("Silhouette", color="#0a0a0a")
ax1.axvline(5, color="#fbbf24", lw=2, alpha=0.7)
ax1.set_title(f"Elbow at k=5, silhouette {sils[3]:.2f}")
plt.tight_layout(); plt.show()


The elbow sits between k=4 and k=5. Silhouette at k=5 is the highest value in the range above k=2. k=5 earns the pick both ways.

A silhouette in the 0.2 to 0.3 band is common for socio-economic count data because the archetypes aren't cleanly separated spheres — they blend at the edges, which is the honest answer for real-world channels. A silhouette above 0.5 on data like this would suggest oversimplification, not success.


## Fit k=5 and profile the clusters

The profile table below summarises each cluster by the median of its features. Medians beat means for this kind of long-tailed data — a single outlier inside a cluster can pull the mean off-centre.


In [ ]:
km5 = KMeans(n_clusters=5, random_state=42, n_init=10).fit(X_std)
df_.loc[Xc.index, "cluster_raw"] = km5.labels_

# Remap to stable archetype ordering for the writeup
tmp = df_.loc[Xc.index, ["subscribers", "views_per_sub", "uploads", "cluster_raw"]]
prof = tmp.groupby("cluster_raw").agg(subs=("subscribers", "median"), vps=("views_per_sub", "median"), ups=("uploads", "median"))
music = prof["ups"].idxmin()
machines = prof["ups"].idxmax()
rest = prof.drop([music, machines])
low_eng = rest["vps"].idxmin()
rest = rest.drop(low_eng)
mega = rest["subs"].idxmax()
mainstream = rest["subs"].idxmin()
stable = {mega: 0, mainstream: 1, low_eng: 2, music: 3, machines: 4}
df_.loc[Xc.index, "cluster"] = df_.loc[Xc.index, "cluster_raw"].map(stable).astype(int)

labels_map = {0: "Mega-scale", 1: "Mainstream", 2: "Low engagement", 3: "Music-video", 4: "Upload machines"}
profile = df_.groupby("cluster")[feats].median().round(1)
profile["count"] = df_.groupby("cluster").size()
profile["label"] = profile.index.map(labels_map)
profile[["label", "subscribers", "video views", "uploads", "views_per_sub", "views_per_upload", "count"]]


Reading the table: cluster 0 has the highest median subscribers at 36.5M, cluster 3 has twelve median uploads (music-video channels live on a handful of major releases), cluster 4 pushes past 10,000 uploads (news networks and content farms), and cluster 2's views-per-subscriber of 133 is roughly a quarter of what every other cluster produces.

That last row is the most interesting finding. Low views-per-subscriber with high subscriber counts means the subscriber base was acquired historically and isn't watching new content. Kids'-content channels that rode YouTube's 2015-2020 recommendation algorithm sit heavily here, as do channels that mass-accumulated subscribers through short-form content that doesn't retain.


## Exemplar channels per cluster

Profiles are abstract. The segmentation earns its pay when the top few channels in each cluster land where the labels predict.


In [ ]:
for c in sorted(df_["cluster"].dropna().unique()):
    sub = df_.loc[df_["cluster"] == c].sort_values("subscribers", ascending=False)
    top = sub["Youtuber"].head(6).dropna().astype(str).tolist()
    print(f"Cluster {int(c)} — {labels_map[int(c)]}:  {', '.join(top)}")


Cluster 0 pulls the names a reader would expect at the top of the platform: T-Series, MrBeast, Cocomelon, SET India. Cluster 3 returns major-label music channels. Cluster 4 returns news networks and short-clip compilation channels. The segmentation maps to reality.


## Visualise the clusters on two interpretable axes

Subscribers and views-per-subscriber are the two axes that separate the clusters most cleanly. The full six-feature distance drives the fit, but plotting two dimensions gives a reader something to look at.


In [ ]:
colors = ["#ef4444", "#0a0a0a", "#fbbf24", "#f87171", "#737373"]
fig, ax = plt.subplots(figsize=(11, 6))
for c in sorted(df_["cluster"].dropna().unique()):
    sub = df_.loc[df_["cluster"] == c]
    ax.scatter(sub["subscribers"]/1e6, sub["views_per_sub"], s=42, alpha=0.75,
               color=colors[int(c)], edgecolor="#0a0a0a", linewidth=0.3,
               label=f"{labels_map[int(c)]} (n={len(sub)})")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Subscribers (millions, log)")
ax.set_ylabel("Views per subscriber (log)")
ax.set_title("Five archetypes on the subs × views/sub plane")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout(); plt.show()


The mega-scale cluster sits upper-right. The low-engagement cluster sits visibly below every other group along the y-axis. The music-video and upload-machines clusters overlap here — they only separate when uploads is added as a third dimension, which the log-standardised fit does.


## Country and category distributions

These are descriptive views of the dataset rather than model outputs. They anchor the segmentation in the real-world composition of the top-995.


In [ ]:
top_countries = df_["Country"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.bar(top_countries.index, top_countries.values, color="#737373")
for i in range(min(3, len(bars))):
    bars[i].set_color("#ef4444")
ax.set_xticklabels(top_countries.index, rotation=30, ha="right")
ax.set_title(f"US tops the list at {top_countries.iloc[0]} of {len(df_)} channels")
ax.set_ylabel("Channels")
plt.tight_layout(); plt.show()


In [ ]:
top_cat = df_["category"].value_counts().head(10)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(top_cat.index[::-1], top_cat.values[::-1], color="#737373")
bars[-1].set_color("#ef4444")
bars[-2].set_color("#fbbf24")
ax.set_xlabel("Channels")
ax.set_title(f"Entertainment ({top_cat.iloc[0]}) and Music ({top_cat.iloc[1]}) dominate")
plt.tight_layout(); plt.show()


## Category earnings (with a warning)

The earnings columns in this dataset are Social Blade band estimates: lowest-yearly and highest-yearly. The median band width across the top-995 is a factor of 16 between floor and ceiling — that is an order-of-magnitude uncertainty, not a rounding error. Social Blade computes from view count under an assumed CPM. Actual creator revenue depends on sponsorships, merch, and content type in ways Social Blade cannot observe.

The category ranking below is therefore meaningful as *ordering*. The absolute dollar levels are not.


In [ ]:
band = df_.loc[df_["lowest_yearly_earnings"] > 0, "highest_yearly_earnings"] / df_.loc[df_["lowest_yearly_earnings"] > 0, "lowest_yearly_earnings"]
print(f"Social Blade high/low ratio — median: {band.median():.1f}x,  p90: {band.quantile(0.9):.1f}x")

cat_earn = df_.groupby("category")["mid_yearly_earnings"].median().sort_values(ascending=False).head(10)
fig, ax = plt.subplots(figsize=(9.5, 5))
bars = ax.barh(cat_earn.index[::-1], cat_earn.values[::-1]/1e6, color="#737373")
bars[-1].set_color("#ef4444")
ax.set_xlabel("Median midpoint (USD millions)")
ax.set_title(f"{cat_earn.index[0]} category leads at ${cat_earn.iloc[0]/1e6:.2f}M median midpoint")
plt.tight_layout(); plt.show()


## Findings

Five clusters cover the top-995 YouTube channels cleanly after log-standardisation. Mega-scale (n=135) sits distinctly above every other group on subscribers. The mainstream cluster (n=367) is the largest and captures the bulk of the 10M-30M subscriber range. Low engagement (n=158) is defined by views-per-subscriber of 133 against a subscriber floor of 14.9M — that ratio is the single sharpest axis in the segmentation. Music-video (n=114) and upload-machines (n=175) sit at opposite ends of the uploads axis, both with comparable subscriber counts but radically different content strategies.

The elbow and silhouette agree on k=5. Silhouette lands in the 0.2-0.3 band, which is honest for this kind of socio-economic count data.

Social Blade earnings estimates have a median high-to-low band width of 16x — treat absolute dollars accordingly.


## Try this next

Refit with an alternative feature set — drop subscribers and work only on engagement ratios (views-per-sub, views-per-upload, uploads-per-sub). Does the segmentation still recover the music-video and upload-machine archetypes without the scale signal, or do they collapse? That answers whether the archetypes are about content strategy or about size.
